In [14]:
from datasets import load_dataset

In [15]:
ds = load_dataset("wikitext", "wikitext-2-v1")
train_texts = [x["text"] for x in ds["train"]]

In [16]:
# Basic cleaning: drop empties, deduplicate exact lines, remove <unk>, normalize space
def clean_line(s: str) -> str:
    s = s.replace("<unk>", "").strip()
    s = " ".join(s.split())  # collapse whitespace
    return s

cleaned = [clean_line(t) for t in train_texts if t and t.strip()]
cleaned = list(dict.fromkeys(cleaned))  # simple dedup while preserving order

len(cleaned), cleaned[0][:120]

(21330, '= Valkyria Chronicles III =')

In [17]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Sequence, NFKC, Lowercase, Strip
from tokenizers.processors import TemplateProcessing

# Create tokenizer with BPE model
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.normalizer = Sequence([NFKC(), Lowercase(), Strip()])
tokenizer.pre_tokenizer = Whitespace()

special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
trainer = BpeTrainer(
    vocab_size=30000,  # adjust if you want 20k/50k etc.
    special_tokens=special_tokens,
    show_progress=True
)

# Tokenizers expects an iterator of files or lines; we’ll pass lines
tokenizer.train_from_iterator(cleaned, trainer=trainer)
print("Vocab size:", tokenizer.get_vocab_size())


Vocab size: 30000


In [18]:
tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B [SEP]",
    special_tokens=[
        ("[CLS]", tokenizer.token_to_id("[CLS]")),
        ("[SEP]", tokenizer.token_to_id("[SEP]")),
    ],
)


In [19]:
import os
from transformers import PreTrainedTokenizerFast

SAVE_DIR = "wikitext2_bpe"
os.makedirs(SAVE_DIR, exist_ok=True)   # ✅ ensures the folder exists

# Save tokenizer.json
tokenizer.save(os.path.join(SAVE_DIR, "tokenizer.json"))

# Wrap with HF-compatible fast tokenizer
hf_tok = PreTrainedTokenizerFast(
    tokenizer_file=os.path.join(SAVE_DIR, "tokenizer.json"),
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

# Save full HF tokenizer package (tokenizer.json + tokenizer_config.json + special tokens)
hf_tok.save_pretrained(SAVE_DIR)

print(f"Tokenizer saved in {SAVE_DIR}/")


Tokenizer saved in wikitext2_bpe/


In [20]:
import numpy as np

val_texts = [clean_line(x["text"]) for x in ds["validation"] if x["text"].strip()]

def encode_count(t):
    return len(hf_tok.encode(t, add_special_tokens=False))

non_empty = [t for t in val_texts if t]
avg_toks = np.mean([encode_count(t) for t in non_empty])

avg_chars = np.mean([len(t) for t in non_empty])
compression = avg_chars / avg_toks

def roundtrip_ok(t):
    ids = hf_tok.encode(t, add_special_tokens=False)
    dec = hf_tok.decode(ids, skip_special_tokens=True)
    return clean_line(t) == clean_line(dec)

consistency = np.mean([roundtrip_ok(t) for t in non_empty]) * 100.0

print({
    "vocab_size": hf_tok.vocab_size,
    "avg_tokens_per_sentence": float(avg_toks),
    "compression_ratio_chars_per_token": float(compression),
    "roundtrip_consistency_%": float(consistency),
})


{'vocab_size': 30000, 'avg_tokens_per_sentence': 83.86143843965867, 'compression_ratio_chars_per_token': 5.038704738277863, 'roundtrip_consistency_%': 5.4449410808614385}


In [21]:
s = "Byte Pair Encoding learns common subwords like 'internationalization' -> 'international' + 'ization'."

# encode returns a list of token IDs
ids = hf_tok.encode(s)

# decode takes a list of IDs back into text
decoded = hf_tok.decode(ids)

print(ids[:20])      # first 20 IDs
print(decoded)       # reconstructed text


[2, 297, 302, 2176, 28293, 9476, 1266, 697, 3617, 711, 11, 1445, 2111, 11, 17, 34, 11, 1445, 11, 15]
[CLS] by te pair encoding learns common sub words like ' international ization ' - > ' international ' + ' ization ' . [SEP]


In [22]:
tokens = hf_tok.convert_ids_to_tokens(ids)
for t, i in zip(tokens, ids):
    print(f"{i:5d}  {t}")


    2  [CLS]
  297  by
  302  te
 2176  pair
28293  encoding
 9476  learns
 1266  common
  697  sub
 3617  words
  711  like
   11  '
 1445  international
 2111  ization
   11  '
   17  -
   34  >
   11  '
 1445  international
   11  '
   15  +
   11  '
 2111  ization
   11  '
   18  .
    3  [SEP]
